In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ 
f:\mathbb{R} \to \mathbb{R}, \quad y = \text{sign}(x) = \begin{cases}
    -1 & \text{if } x < 0 \\
    0 & \text{if } x = 0 \\
    1 & \text{if } x > 0
\end{cases} 
$$

$$ 
f:\mathbb{R^n} \to \mathbb{R^n}, \quad \mathbf{y} = 
\text{sign}(\mathbf{x}) = 
(f(x_1), f(x_2), \dots, f(x_n))
$$

$$ 
\frac{df}{dx_i} = \begin{cases}
    0 & \text{if } x_i \neq 0 \\
    \text{undefined} & \text{if } x_i = 0
\end{cases} 
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
from approx import approx # type: ignore


def sign(x: tr.Tensor) -> tr.Tensor:
    """`sign` function."""

    x = tr.as_tensor(x)
    
    y = tr.zeros_like(x)
    y[x > 0] = 1
    y[x < 0] = -1
    return y
    

class SignFunction(autograd.Function):
    """Custom autograd function for the `sign`."""

    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        y = sign(x)
        return y

    @staticmethod
    def backward(ctx, grad_output: tr.Tensor) -> tuple[tr.Tensor, ]:
        return (tr.zeros_like(grad_output), )
    

class Sign(nn.Module):
    """Custom module for the `sign`."""

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        y = SignFunction.apply(x)
        return y

In [ ]:
# Unit tests

def test_basic() -> None:
    assert sign(-2) == -1
    assert sign(-1) == -1
    assert sign(0) == 0
    assert sign(1) == +1
    assert sign(2) == +1


def test_gradcheck(samples) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return SignFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)